# 47 · Chỉ bật phim trước ở **những nhãn nó giúp** — hậu kỳ, không huấn luyện lại

`45` đo được: phim trước cho `+0.0029` macro nhưng **`−0.0039` tail**, và phần nó ăn nằm ở nhóm
**cấu trúc** (`+0.0080` sau khi trừ đối chứng). Ba notebook (`44a`/`44b`/`45`) đều thấy nó **hại tail**.

Câu hỏi: bật nó **chỉ ở nhãn nó giúp** thì có vừa giữ lợi vừa tránh hại không.

## Vì sao cross-fitting chứ không phải lát chọn

`45` chỉ `np.save` dự đoán trên **`val`**; dự đoán trên lát chọn không được lưu. Chọn nhãn trên lát
chọn sẽ phải chạy lại GPU.

Thay bằng **cross-fitting theo BỆNH NHÂN trên `val`**:

```
chia val thanh hai nua A/B, CAT THEO subject_id (khong benh nhan nao o ca hai)
chon tap nhan S tren A  ->  cham tren B
chon tap nhan S tren B  ->  cham tren A
```

Không bao giờ chọn và chấm trên **cùng bệnh nhân**. Ước lượng không chệch, tốn vài giây.

Có in kèm **`ORACLE`** (chọn và chấm trên cùng dữ liệu) — đó là **trần rò rỉ**, để biết khoảng cách,
**không** phải con số dùng được.

## Không hard-code danh sách nhãn cấu trúc

Danh sách 10 nhãn cấu trúc trong `C8` của `44b`/`45` là thứ tôi **đọc ra từ kết quả val**. Nhét nó
vào đây là rò rỉ val vào thiết kế — đúng điều kế hoạch cảnh báo với 11 nhãn nhạy EHR của `33` C7.
Ở đây tập nhãn được **chọn bằng dữ liệu**, rồi mới đối chiếu xem có trùng nhóm cấu trúc không.

## C0 · Nạp — chỉ đọc `p_*.npy` đã lưu, không GPU

In [1]:
import os, json, re
import numpy as np, pandas as pd

try:
    from google.colab import drive; drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024'); EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RES47 = os.path.join(OUT_DIR, 'stage4_nb47')
def ensure_res():
    os.makedirs(RES47, exist_ok=True); return RES47
ensure_res(); open(os.path.join(RES47, '_run_info.json'), 'w').write('{"notebook":"47"}')

# hai lan chay DOC LAP — de xem viec tron co LAP LAI khong
RUNS = {'45':  dict(d=os.path.join(OUT_DIR, 'stage4_nb45', 'c1_t1prior_control'), ep=10),
        '44b': dict(d=os.path.join(OUT_DIR, 'stage4_nb44b', 'b2_prior_patch_fix'), ep=6)}
BASE, PRIOR, SEED = 'mldec_t1', 't1_prior', 42

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id']+LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
YM = Y[LAB40].to_numpy('float32')
VA = ORDER.split.to_numpy() == 'val'
SUBV = ORDER.subject_id.to_numpy()[VA]; YV = YM[VA]
print(f'val {VA.sum():,} anh | {len(np.unique(SUBV)):,} benh nhan')

_u = np.unique(SUBV); _rs = np.random.default_rng(0); _rs.shuffle(_u)
_hA = set(_u[:len(_u)//2].tolist())
FA = np.array([s in _hA for s in SUBV]); FB = ~FA
assert not (set(SUBV[FA]) & set(SUBV[FB])), 'hai nua CHUNG benh nhan'
print(f'nua A {FA.sum():,} anh / {len(np.unique(SUBV[FA])):,} BN | '
      f'nua B {FB.sum():,} anh / {len(np.unique(SUBV[FB])):,} BN | khong chung BN')

Mounted at /content/drive
val 39,293 anh | 6,821 benh nhan
nua A 19,600 anh / 3,410 BN | nua B 19,693 anh / 3,411 BN | khong chung BN


## C1 · AP từng nhãn + phép trộn

In [2]:
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, sc = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-sc, kind='stable'); yy, ss = y[r], sc[r]
        tp, fp = np.cumsum(yy), np.cumsum(1-yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp/(tp+fp), tp/n1
        o[j] = float((np.diff(np.r_[0.0, rc])*pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}

def chon(Pb, Pp, msk, eps=0.0):
    '''Tap nhan ma `prior` hon `base` TREN phan `msk`.'''
    d = ap_cols(YV[msk], Pp[msk]) - ap_cols(YV[msk], Pb[msk])
    return np.where(np.nan_to_num(d, nan=-1.0) > eps)[0], d

STRUCT = [l for l in LAB40 if re.search(
    r'pneumothorax|hernia|fracture|mass|nodule|calcif|scoliosis|support device', l, re.I)]
IDX_STR = np.array([LAB40.index(l) for l in STRUCT], dtype=int)
grp = lambda ap: dict(macro=float(np.nanmean(ap)),
                      cau_truc=float(np.nanmean(ap[IDX_STR])), **tiers(ap))
print(f'nhom CAU TRUC ({len(STRUCT)}) — chi de DOI CHIEU, khong dung de chon: {STRUCT}')

nhom CAU TRUC (10) — chi de DOI CHIEU, khong dung de chon: ['Calcification of the Aorta', 'Clavicle Fracture', 'Fracture', 'Hernia', 'Hydropneumothorax', 'Mass', 'Nodule', 'Pneumothorax', 'Rib Fracture', 'Support Devices']


## C2 · Chạy cả hai lần chạy, nhiều ngưỡng `eps`

`eps` là mức cải thiện tối thiểu để một nhãn được bật phim trước. `eps=0` bật mọi nhãn có lợi dù
nhỏ — với 40 nhãn thì gần một nửa sẽ dương do ngẫu nhiên, nên `eps` lớn hơn là **chống nhiễu chọn nhãn**.

Mỗi nhãn lấy **trọn** từ một arm, không pha trộn xác suất ⇒ không có vấn đề calibration giữa hai arm.

In [3]:
rows, SEL = [], {}
for nm, cfg in RUNS.items():
    f = lambda a: os.path.join(cfg['d'], f"p_{a}_s{SEED}_ep{cfg['ep']}.npy")
    if not (os.path.exists(f(BASE)) and os.path.exists(f(PRIOR))):
        print(f'{nm}: khong thay {f(BASE)} hoac {f(PRIOR)} — bo qua'); continue
    Pb, Pp = np.load(f(BASE)), np.load(f(PRIOR))
    assert Pb.shape == (VA.sum(), C), f'{nm}: {Pb.shape} khong khop ({VA.sum()}, {C})'
    rows.append(dict(run=nm, cach='base  (mldec_t1)', n_nhan=0, **grp(ap_cols(YV, Pb))))
    rows.append(dict(run=nm, cach='prior (t1_prior)', n_nhan=C, **grp(ap_cols(YV, Pp))))
    for eps in (0.0, 0.002, 0.005):
        SA, _ = chon(Pb, Pp, FA, eps); SB, _ = chon(Pb, Pp, FB, eps)
        P = Pb.copy()
        if len(SA): P[np.ix_(FB, SA)] = Pp[np.ix_(FB, SA)]    # chon o A -> ap o B
        if len(SB): P[np.ix_(FA, SB)] = Pp[np.ix_(FA, SB)]    # chon o B -> ap o A
        rows.append(dict(run=nm, cach=f'cross-fit eps={eps:.3f}',
                         n_nhan=(len(SA)+len(SB))/2, **grp(ap_cols(YV, P))))
        SEL[(nm, eps)] = (SA, SB)
    So, d_all = chon(Pb, Pp, np.ones(len(YV), bool), 0.0)
    Po = Pb.copy(); Po[:, So] = Pp[:, So]
    rows.append(dict(run=nm, cach='ORACLE (RO RI)', n_nhan=len(So), **grp(ap_cols(YV, Po))))
    SEL[(nm, 'oracle')] = (So, d_all)

T = pd.DataFrame(rows)
print(T.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
ensure_res(); T.to_csv(os.path.join(RES47, 'blend.csv'), index=False)
print('')
print('  DOC: so `cross-fit` voi `base`. Neu no KHONG hon -> phim truoc khong tach bach duoc')
print('       theo nhan, va y tuong nay dong voi gia bang 0.')
print('  `ORACLE` la TRAN RO RI, khong phai con so dung duoc.')

run                cach  n_nhan  macro  cau_truc   head  medium   tail
 45    base  (mldec_t1)  0.0000 0.2463    0.3301 0.4610  0.1735 0.0986
 45    prior (t1_prior) 40.0000 0.2513    0.3369 0.4664  0.1852 0.0932
 45 cross-fit eps=0.000 29.5000 0.2501    0.3346 0.4660  0.1834 0.0922
 45 cross-fit eps=0.002 25.0000 0.2495    0.3330 0.4644  0.1826 0.0927
 45 cross-fit eps=0.005 20.0000 0.2489    0.3331 0.4636  0.1813 0.0935
 45      ORACLE (RO RI) 30.0000 0.2544    0.3421 0.4665  0.1853 0.1042
44b    base  (mldec_t1)  0.0000 0.2513    0.3389 0.4642  0.1784 0.1058
44b    prior (t1_prior) 40.0000 0.2562    0.3450 0.4707  0.1838 0.1080
44b cross-fit eps=0.000 32.0000 0.2557    0.3439 0.4705  0.1836 0.1066
44b cross-fit eps=0.002 30.0000 0.2554    0.3439 0.4702  0.1834 0.1064
44b cross-fit eps=0.005 19.5000 0.2535    0.3392 0.4685  0.1814 0.1041
44b      ORACLE (RO RI) 34.0000 0.2585    0.3510 0.4707  0.1843 0.1156

  DOC: so `cross-fit` voi `base`. Neu no KHONG hon -> phim truoc khong tach 

## C3 · Tập nhãn được chọn có trùng nhóm cấu trúc không

In [4]:
_tail = set(TIER['tail'].tolist())
for nm in RUNS:
    if (nm, 'oracle') not in SEL: continue
    So, d = SEL[(nm, 'oracle')]
    ov = len(set(So.tolist()) & set(IDX_STR.tolist()))
    print(f'=== {nm} ===')
    print(f'  oracle chon {len(So)}/40 nhan | trung nhom CAU TRUC {ov}/{len(IDX_STR)}'
          f' | trong tap chon co {len(set(So.tolist()) & _tail)}/{len(_tail)} nhan TAIL')
    _o = np.argsort(-np.nan_to_num(d, nan=-9))
    print('  len nhieu nhat: ' + ' · '.join(f'{LAB40[j]} {d[j]:+.4f}' for j in _o[:6]))
    print('  tut nhieu nhat: ' + ' · '.join(f'{LAB40[j]} {d[j]:+.4f}' for j in _o[-5:]))
    SA, SB = SEL[(nm, 0.0)]
    print(f'  ON DINH: nua A chon {len(SA)}, nua B chon {len(SB)}, CHUNG {len(set(SA.tolist())&set(SB.tolist()))}')
    print('    (chung it -> tap nhan chu yeu la NHIEU, va cross-fit se khong hon base)')
    print('')

=== 45 ===
  oracle chon 30/40 nhan | trung nhom CAU TRUC 9/10 | trong tap chon co 4/11 nhan TAIL
  len nhieu nhat: Subcutaneous Emphysema +0.0579 · Hernia +0.0281 · Azygos Lobe +0.0267 · Hydropneumothorax +0.0266 · Pneumothorax +0.0222 · Fibrosis +0.0207
  tut nhieu nhat: Pneumomediastinum -0.0099 · Infarction -0.0130 · Pneumoperitoneum -0.0190 · Pleural Other -0.0209 · Clavicle Fracture -0.0522
  ON DINH: nua A chon 26, nua B chon 33, CHUNG 23
    (chung it -> tap nhan chu yeu la NHIEU, va cross-fit se khong hon base)

=== 44b ===
  oracle chon 34/40 nhan | trung nhom CAU TRUC 9/10 | trong tap chon co 8/11 nhan TAIL
  len nhieu nhat: Hydropneumothorax +0.0265 · Pneumothorax +0.0237 · Fracture +0.0210 · Pneumoperitoneum +0.0197 · Pneumomediastinum +0.0185 · Subcutaneous Emphysema +0.0175
  tut nhieu nhat: Emphysema -0.0003 · Lobar Atelectasis -0.0003 · Granuloma -0.0083 · Round(ed) Atelectasis -0.0234 · Clavicle Fracture -0.0601
  ON DINH: nua A chon 33, nua B chon 31, CHUNG 27
    (c

## C4 · Ghi `REVIEW_PRIOR_PERLABEL.md`

In [5]:
L = []; w = L.append
w('# REVIEW · bật phim trước theo từng nhãn (`nb47`, hậu kỳ, không huấn luyện lại)'); w('')
w('Tập nhãn chọn bằng **cross-fitting theo bệnh nhân** trên `val` (chọn ở nửa này, chấm ở nửa kia). '
  '`45` chỉ lưu dự đoán trên `val` nên không dùng lát chọn được.'); w('')
w('| lần chạy | cách | #nhãn | macro | cấu trúc | head | medium | tail |')
w('|---|---|---|---|---|---|---|---|')
for _, r in T.iterrows():
    w(f'| `{r.run}` | {r.cach} | {r.n_nhan:.0f} | {r.macro:.4f} | {r.cau_truc:.4f} | '
      f'{r["head"]:.4f} | {r["medium"]:.4f} | {r["tail"]:.4f} |')
w('')
w('**`ORACLE` là TRẦN RÒ RỈ** (chọn và chấm trên cùng dữ liệu) — in ra để biết khoảng cách, '
  '**không** phải con số dùng được.'); w('')
w('## Những gì kết quả này KHÔNG trả lời'); w('')
w('- **Trộn hậu kỳ ≠ huấn luyện có định tuyến.** Ở đây mỗi nhãn lấy trọn từ một arm. Một mô hình '
  'học định tuyến ở **tầng 2** (nối phim trước vào K/V của 40 query lớp) có thể tốt hơn — hoặc tệ '
  'hơn. Chưa đo.')
w('- **Một seed mỗi lần chạy.** Tập nhãn được chọn tự nó có nhiễu; cột `#nhãn` và độ chồng lấn giữa '
  'hai nửa (`C3`) cho biết nó ổn định tới đâu.')
w('- **Nhóm cấu trúc chỉ dùng để ĐỐI CHIẾU**, không dùng để chọn — nó được đọc ra từ kết quả val ở '
  '`44a`/`44b`, nên dùng nó để chọn là rò rỉ.')
TXT = '\n'.join(L)
ensure_res(); open(os.path.join(RES47, 'REVIEW_PRIOR_PERLABEL.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)

# REVIEW · bật phim trước theo từng nhãn (`nb47`, hậu kỳ, không huấn luyện lại)

Tập nhãn chọn bằng **cross-fitting theo bệnh nhân** trên `val` (chọn ở nửa này, chấm ở nửa kia). `45` chỉ lưu dự đoán trên `val` nên không dùng lát chọn được.

| lần chạy | cách | #nhãn | macro | cấu trúc | head | medium | tail |
|---|---|---|---|---|---|---|---|
| `45` | base  (mldec_t1) | 0 | 0.2463 | 0.3301 | 0.4610 | 0.1735 | 0.0986 |
| `45` | prior (t1_prior) | 40 | 0.2513 | 0.3369 | 0.4664 | 0.1852 | 0.0932 |
| `45` | cross-fit eps=0.000 | 30 | 0.2501 | 0.3346 | 0.4660 | 0.1834 | 0.0922 |
| `45` | cross-fit eps=0.002 | 25 | 0.2495 | 0.3330 | 0.4644 | 0.1826 | 0.0927 |
| `45` | cross-fit eps=0.005 | 20 | 0.2489 | 0.3331 | 0.4636 | 0.1813 | 0.0935 |
| `45` | ORACLE (RO RI) | 30 | 0.2544 | 0.3421 | 0.4665 | 0.1853 | 0.1042 |
| `44b` | base  (mldec_t1) | 0 | 0.2513 | 0.3389 | 0.4642 | 0.1784 | 0.1058 |
| `44b` | prior (t1_prior) | 40 | 0.2562 | 0.3450 | 0.4707 | 0.1838 | 0.1080 |
| `44b` | cross-fit eps=